# 📚 Research Paper Answer Bot — Capstone Project

An end-to-end Retrieval-Augmented Generation (RAG) system built over 8 seminal GenAI and Large Language Model (LLM) research papers.

### Key Capabilities:
- **Data Ingestion**: Automated arXiv downloading & PDF processing (PyPDF vs. PDFPlumber)
- **Chunking**: Comparative evaluation of Fixed-size, Recursive, and Semantic chunking
- **Embeddings**: Dual-model benchmark (Open-source BGE-small vs. Commercial OpenAI text-embedding-3-small experiment)
- **Vector Store**: Local persistent vector index with ChromaDB
- **Retrieval**: 5 advanced strategies (Dense, MMR, Hybrid BM25+Dense, Cross-Encoder Reranking, Multi-Query)
- **LCEL Pipeline**: Strict grounded prompt engineering with exact paper & page citations
- **LLM Generation**: Google Gemini API (`gemini-2.5-flash`)
- **Evaluation**: 10 test questions with retrieval hit-rate analysis & RAGAS metrics
- **Conversational Memory**: Sliding-window multi-turn Q&A support
- **Streamlit UI**: Full-fledged web chat interface with source expanders

> **Architecture Note**: OpenAI API generation was replaced with Google Gemini API to avoid dependency on the exhausted OpenAI API quota while preserving the entire RAG pipeline and retrieval architecture.

## 1. Setup & Environment Dependencies

In [ ]:
import sys
from pathlib import Path

# Ensure project modules are importable
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import PAPERS_METADATA, GEMINI_API_KEY, OPENAI_API_KEY
print(f"Loaded metadata for {len(PAPERS_METADATA)} research papers.")
print(f"Google Gemini API Key configured: {bool(GEMINI_API_KEY)}")
print(f"OpenAI API Key configured (for embedding experiment): {bool(OPENAI_API_KEY)}")

## 2. Step 1: Downloading Seminal Research Papers

In [ ]:
from src.data_loader import download_papers

downloaded_files = download_papers()
for key, path in downloaded_files.items():
    print(f"  • {key}: {path.name}")

## 3. Step 2: Document Loading & Inspection

In [ ]:
from src.data_loader import load_papers_pypdf, inspect_documents

documents = load_papers_pypdf()
inspection_df = inspect_documents(documents)
inspection_df.head(10)

## 4. Step 3: Text Chunking Strategies & Analysis

In [ ]:
from src.chunking import chunk_recursive, compare_chunking_strategies, get_chunking_stats

chunks = chunk_recursive(documents)
stats_df = get_chunking_stats(chunks)
stats_df.head()

## 5. Step 4: Embedding Model Setup

In [ ]:
from src.embeddings import get_hf_embeddings

hf_embeddings = get_hf_embeddings()
print("HuggingFace BGE embeddings model successfully initialized.")

## 6. Step 5: Vector Database Setup (ChromaDB)

In [ ]:
from src.vectorstore import setup_hf_vectorstore, validate_retrieval

hf_vectorstore = setup_hf_vectorstore(chunks)
validate_retrieval(hf_vectorstore)

## 7. Step 6: Advanced Retrieval Strategies

In [ ]:
from src.retrieval import get_dense_retriever, get_mmr_retriever, get_hybrid_retriever

dense_retriever = get_dense_retriever(hf_vectorstore)
mmr_retriever = get_mmr_retriever(hf_vectorstore)
hybrid_retriever = get_hybrid_retriever(hf_vectorstore, chunks)

test_query = "What is the self-attention mechanism?"
print("--- Hybrid Search Results ---")
for doc in hybrid_retriever.invoke(test_query)[:3]:
    print(f"• [{doc.metadata.get('paper_title')}] Page {doc.metadata.get('page')}: {doc.page_content[:120]}...")

## 8. Step 7: Multi-Turn Conversational Memory & Gemini LLM Demo

In [ ]:
from src.memory import ConversationalRAG

conv_rag = ConversationalRAG(retriever=hybrid_retriever)
print("Conversational RAG instance initialized with Google Gemini LLM and sliding memory window (k=5).")

## 9. Launching the Interactive Web UI

To launch the Streamlit chat interface, run the following command in your terminal:
```bash
streamlit run streamlit_app.py
```